# Part 07: Mixture of Experts: Routing Tokens to Specialists

> **Previous context**: Part 06 improved the dense Transformer block. MoE changes the feed-forward part so not every token uses the same expert.
> **Goal for this part**: Implement a small MoE layer, top-k routing, and load-balancing intuition.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why MoE exists

Dense models use all parameters for every token. MoE increases total parameters while activating only a few experts per token.

## 1. Router intuition

A router scores experts for each token, chooses the top experts, and combines their outputs with gate weights.

## 2. Load balance

If one expert receives all tokens, the system wastes capacity. Auxiliary losses encourage more even expert usage.

## 3. Trade-off

MoE can increase model capacity efficiently, but it makes training, communication, batching, and serving more complex.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] The router chooses experts per token.
- [ ] Only selected experts run, so active compute stays smaller than total parameters.
- [ ] Load balancing prevents expert collapse.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
class MoELayer(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, num_experts=8, top_k=2, expert_dim=None):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        expert_dim = expert_dim or 4 * d_model
        
        # Teaching note: follow this line to see the main step.
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        
        # Teaching note: follow this line to see the main step.
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, d_model)
            )
            for _ in range(num_experts)
        ])
    
    def forward(self, x):
        'Read the values printed above and connect them to the concept in this cell.'
        batch_size, seq_len, d_model = x.shape
        
        # Teaching note: follow this line to see the main step.
        gate_logits = self.gate(x)  # [batch, seq_len, num_experts]
        
        # Teaching note: follow this line to see the main step.
        top_k_logits, top_k_indices = torch.topk(gate_logits, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)  # Teaching note: follow this line to see the main step.
        
        # Teaching note: follow this line to see the main step.
        output = torch.zeros_like(x)
        
        for b in range(batch_size):
            for s in range(seq_len):
                token = x[b, s]  # [d_model]
                for k in range(self.top_k):
                    expert_idx = top_k_indices[b, s, k].item()
                    weight = top_k_weights[b, s, k]
                    expert_out = self.experts[expert_idx](token.unsqueeze(0)).squeeze(0)
                    output[b, s] += weight * expert_out
        
        return output

print('Read the values printed above and connect them to the concept in this cell.')
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [3]:
# Teaching note: follow this line to see the main step.
moe = MoELayer(d_model=16, num_experts=8, top_k=2)

# Teaching note: follow this line to see the main step.
x = torch.randn(1, 3, 16)

# Teaching note: follow this line to see the main step.
with torch.no_grad():
    gate_scores = moe.gate(x).squeeze(0)  # [3, 8]
    top_k_vals, top_k_idx = torch.topk(gate_scores, 2, dim=-1)

print('Read the values printed above and connect them to the concept in this cell.')
print()
for i in range(3):
    experts = top_k_idx[i].tolist()
    weights = F.softmax(top_k_vals[i], dim=-1).tolist()
    print(f"Token {i}Read the values printed above and connect them to the concept in this cell.{experts}Read the values printed above and connect them to the concept in this cell.{[f'{w:.2f}' for w in weights]}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [4]:
# Teaching note: follow this line to see the main step.
class TinyDenseBlock(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model=16, num_heads=2, ffn_dim=64):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class TinyMoEBlock(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model=16, num_heads=2, num_experts=4, top_k=2, expert_dim=64):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.moe = MoELayer(d_model, num_experts=num_experts, top_k=top_k, expert_dim=expert_dim)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.norm1(x + attn_out)
        moe_out = self.moe(x)
        x = self.norm2(x + moe_out)
        return x


dense_block = TinyDenseBlock()
moe_block = TinyMoEBlock(num_experts=4, top_k=2)

print('Read the values printed above and connect them to the concept in this cell.')
print(dense_block)
print()
print("=== MoE Transformer Block ===")
print(moe_block)


Read the values printed above and connect them to the concept in this cell.TinyDenseBlock(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (ffn): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
  )
  (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
)

=== MoE Transformer Block ===
TinyMoEBlock(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (moe): MoELayer(
    (gate): Linear(in_features=16, out_features=4, bias=False)
    (experts): ModuleList(
      (0-3): 4 x Sequential(
        (0): Linear(in_features=16, out_features=64, bias=True)
        (1): ReLU()
        (2): Linear(in_fe

Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [5]:
# Teaching note: follow this line to see the main step.
trace_x = torch.randn(1, 3, 16)

print("=== Dense Block Trace ===")
with torch.no_grad():
    dense_attn_out, _ = dense_block.self_attn(trace_x, trace_x, trace_x, need_weights=False)
    dense_after_attn = dense_block.norm1(trace_x + dense_attn_out)
    dense_ffn_out = dense_block.ffn(dense_after_attn)
    dense_out = dense_block.norm2(dense_after_attn + dense_ffn_out)

print(f"input:        {tuple(trace_x.shape)}")
print(f"attention:    {tuple(dense_attn_out.shape)}")
print(f"single FFN:   {tuple(dense_ffn_out.shape)}")
print(f"output:       {tuple(dense_out.shape)}")

print()
print("=== MoE Block Trace ===")
with torch.no_grad():
    moe_attn_out, _ = moe_block.self_attn(trace_x, trace_x, trace_x, need_weights=False)
    moe_after_attn = moe_block.norm1(trace_x + moe_attn_out)
    gate_logits = moe_block.moe.gate(moe_after_attn)
    top_vals, top_idx = torch.topk(gate_logits, moe_block.moe.top_k, dim=-1)
    moe_out_raw = moe_block.moe(moe_after_attn)
    moe_out = moe_block.norm2(moe_after_attn + moe_out_raw)

print(f"input:        {tuple(trace_x.shape)}")
print(f"attention:    {tuple(moe_attn_out.shape)}")
print(f"gate logits:  {tuple(gate_logits.shape)}Read the values printed above and connect them to the concept in this cell.")
print(f"top-k index:  {tuple(top_idx.shape)}Read the values printed above and connect them to the concept in this cell.")
print(f"MoE FFN:      {tuple(moe_out_raw.shape)}")
print(f"output:       {tuple(moe_out.shape)}")
print()
print('Read the values printed above and connect them to the concept in this cell.')
for token_i, experts in enumerate(top_idx[0].tolist()):
    print(f"token {token_i}: experts {experts}")


=== Dense Block Trace ===
input:        (1, 3, 16)
attention:    (1, 3, 16)
single FFN:   (1, 3, 16)
output:       (1, 3, 16)

=== MoE Block Trace ===
input:        (1, 3, 16)
attention:    (1, 3, 16)
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.MoE FFN:      (1, 3, 16)
output:       (1, 3, 16)

Read the values printed above and connect them to the concept in this cell.token 0: experts [3, 2]
token 1: experts [1, 3]
token 2: experts [3, 2]


In [6]:
# Teaching note: follow this line to see the main step.
import inspect
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

from transformers import Qwen2MoeConfig, Qwen2MoeForCausalLM
from transformers import MixtralConfig, MixtralForCausalLM

qwen_cfg = Qwen2MoeConfig(
    vocab_size=128,
    hidden_size=32,
    intermediate_size=64,
    moe_intermediate_size=64,
    shared_expert_intermediate_size=64,
    num_hidden_layers=1,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_experts=4,
    num_experts_per_tok=2,
)
qwen_moe = Qwen2MoeForCausalLM(qwen_cfg)
qwen_layer = qwen_moe.model.layers[0]

print("=== Qwen2-MoE decoder layer ===")
print(qwen_layer)

print()
print('Read the values printed above and connect them to the concept in this cell.')
for name, param in qwen_layer.mlp.named_parameters():
    if name.startswith("gate") or name.startswith("experts") or name.startswith("shared"):
        print(f"{name:32s} {tuple(param.shape)}")

mixtral_cfg = MixtralConfig(
    vocab_size=128,
    hidden_size=32,
    intermediate_size=64,
    num_hidden_layers=1,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_local_experts=4,
    num_experts_per_tok=2,
)
mixtral_moe = MixtralForCausalLM(mixtral_cfg)
mixtral_layer = mixtral_moe.model.layers[0]

print()
print("=== Mixtral decoder layer ===")
print(mixtral_layer)

print()
print('Read the values printed above and connect them to the concept in this cell.')
for name, param in mixtral_layer.mlp.named_parameters():
    if name.startswith("gate") or name.startswith("experts"):
        print(f"{name:32s} {tuple(param.shape)}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
source = inspect.getsource(type(qwen_layer).forward).splitlines()
for line in source:
    if "self_attn" in line or "mlp" in line or "layernorm" in line:
        print(line)

print()
print('Read the values printed above and connect them to the concept in this cell.')
source = inspect.getsource(type(qwen_layer.mlp).forward).splitlines()
for line in source:
    if "gate" in line or "expert" in line or "router" in line or "top" in line:
        print(line)

input_ids = torch.randint(0, 128, (1, 6))
with torch.no_grad():
    qwen_out = qwen_moe(input_ids=input_ids, output_router_logits=True)

router_logits = qwen_out.router_logits[0]
top_vals, top_idx = torch.topk(router_logits, qwen_cfg.num_experts_per_tok, dim=-1)

print()
print("=== Qwen2-MoE router trace ===")
print(f"input_ids:      {tuple(input_ids.shape)}")
print(f"logits:         {tuple(qwen_out.logits.shape)}")
print(f"router logits:  {tuple(router_logits.shape)}")
print(f"top-k experts:  {tuple(top_idx.shape)}")
print()
print('Read the values printed above and connect them to the concept in this cell.')
for token_i, experts in enumerate(top_idx[:6].tolist()):
    print(f"token {token_i}: experts {experts}")


=== Qwen2-MoE decoder layer ===
Qwen2MoeDecoderLayer(
  (self_attn): Qwen2MoeAttention(
    (q_proj): Linear(in_features=32, out_features=32, bias=True)
    (k_proj): Linear(in_features=32, out_features=32, bias=True)
    (v_proj): Linear(in_features=32, out_features=32, bias=True)
    (o_proj): Linear(in_features=32, out_features=32, bias=False)
  )
  (mlp): Qwen2MoeSparseMoeBlock(
    (gate): Qwen2MoeTopKRouter()
    (experts): Qwen2MoeExperts(
      (act_fn): SiLUActivation()
    )
    (shared_expert): Qwen2MoeMLP(
      (gate_proj): Linear(in_features=32, out_features=64, bias=False)
      (up_proj): Linear(in_features=32, out_features=64, bias=False)
      (down_proj): Linear(in_features=64, out_features=32, bias=False)
      (act_fn): SiLUActivation()
    )
    (shared_expert_gate): Linear(in_features=32, out_features=1, bias=False)
  )
  (input_layernorm): Qwen2MoeRMSNorm((32,), eps=1e-06)
  (post_attention_layernorm): Qwen2MoeRMSNorm((32,), eps=1e-06)
)

Read the values printed

---

1. .2. .3. .4. .5. Loss6. .7. .8. .
.